# Compute mutation rates for subset trees

Read mutation counts for host, geographic, and temporal subsets and compute basic rates. Host counts come from the single host-stratified file (which carries a `host` column); geographic and temporal counts come from per-subset trees.

In [1]:
import os
import glob
import yaml
import pandas as pd
import numpy as np

In [2]:
# Load config
with open('../config.yaml') as f:
    config = yaml.safe_load(f)

segments = config['segments']
ha_subtypes = config['ha_subtypes']
na_subtypes = config['na_subtypes']
host_groups = set(config['host_groups'])

# Geographic and temporal subsets still come from per-subset trees.
# Host counts come from the single host-stratified file produced from the global tree.
non_host_subset_groups = {
    'geographic': config['geographic_groups'],
    'temporal': config['temporal_groups'],
}
print('Host groups:', sorted(host_groups))
print('Non-host subset groups:', non_host_subset_groups)

Host groups: ['avian', 'human', 'swine']
Non-host subset groups: {'geographic': ['north_america', 'europe', 'asia'], 'temporal': ['early', 'late']}


In [3]:
def get_subtypes_for_segment(segment):
    if segment == 'HA':
        return ha_subtypes
    elif segment == 'NA':
        return na_subtypes
    else:
        return ['all']


def read_counts(f, segment, subtype, subset, subset_type):
    """Read a mutation_counts.csv file and add metadata columns."""
    df = pd.read_csv(f, keep_default_na=False, low_memory=False)
    df = df.replace('', np.nan)
    df['subtype'] = subtype
    df['segment'] = segment
    df['segment_subtype'] = segment + '_' + subtype
    df['segment_length'] = df['site'].max() - df['site'].min()
    df['subset'] = subset
    df['subset_type'] = subset_type
    return df

In [4]:
# Read mutation counts from all subset trees
dfs = []
for segment in segments:
    for subtype in get_subtypes_for_segment(segment):

        # Host subsets: single file with a 'host' column
        host_file = f'../results/{segment}/{subtype}/host_stratified/mutation_counts.csv'
        if not os.path.exists(host_file):
            print(f'WARNING: missing {host_file}')
        else:
            host_df = pd.read_csv(host_file, keep_default_na=False, low_memory=False)
            host_df = host_df.replace('', np.nan)
            host_df['subtype'] = subtype
            host_df['segment'] = segment
            host_df['segment_subtype'] = segment + '_' + subtype
            host_df['segment_length'] = host_df['site'].max() - host_df['site'].min()
            host_df = host_df[host_df['host'].isin(host_groups)].copy()
            host_df['subset'] = host_df['host']
            host_df['subset_type'] = 'host'
            host_df = host_df.drop(columns=['host'])
            dfs.append(host_df)

        # Geographic and temporal subsets: per-subset tree files
        for subset_type, groups in non_host_subset_groups.items():
            for group in groups:
                f = f'../results/{segment}/{subtype}/{group}/mutation_counts.csv'
                if not os.path.exists(f):
                    print(f'WARNING: missing {f}')
                    continue
                df = read_counts(f, segment, subtype, group, subset_type)
                dfs.append(df)

counts_df = (
    pd.concat(dfs)
    .rename(columns={'parent_motif': 'motif'})
)

# Compute evolutionary opportunities and rates
counts_df['evo_opp'] = counts_df['syn_branch_length'] / counts_df['segment_length']
counts_df['rate'] = counts_df['actual_count'] / counts_df['evo_opp']

# Remove rows with incomplete motifs
counts_df = counts_df[
    (counts_df['motif'].notnull()) &
    (counts_df['motif'].str.len() == 3)
]

# Save
counts_df.sort_values('mut_type', inplace=True)
counts_df.to_csv('../results/subset_counts.csv', index=False)

print(f'Total rows: {len(counts_df):,}')
print(f'Subsets: {counts_df["subset"].unique()}')
counts_df.head()

Total rows: 4,008,615
Subsets: <StringArray>
['europe', 'early', 'late', 'asia', 'north_america', 'avian', 'human',
 'swine']
Length: 8, dtype: str


,site,nt_mut,wt_nt,mut_nt,gene,codon_position,codon_site,wt_codon,mut_codon,wt_aa,...,mut_class,mut_type,subtype,segment,segment_subtype,segment_length,subset,subset_type,evo_opp,rate
16860,490,A490C,A,C,HA,1,164,AAC,CAC,N,...,nonsynonymous,AC,H1,HA,HA_H1,1698,europe,geographic,11.656066,0.0
20228,1392,A1392C,A,C,HA,3,464,CTA,CTC,L,...,synonymous,AC,H7,HA,HA_H7,1680,early,temporal,0.008333,0.0
30265,1514,A1514C,A,C,HA,2,505,AAG,ACG,K,...,nonsynonymous,AC,H9,HA,HA_H9,1680,late,temporal,0.007143,0.0
30264,1514,A1514C,A,C,HA,2,505,AAA,ACA,K,...,nonsynonymous,AC,H9,HA,HA_H9,1680,late,temporal,2.028571,0.0
25983,1130,A1130C,A,C,NP,2,377,AAC,ACC,N,...,nonsynonymous,AC,all,NP,NP_all,1494,asia,geographic,9.115797,0.0
